# MOSEI Preprocessing — **Person 4 of 4** (shard 3)

You process **your quarter** of the 6,277 CMU-MOSEI clips → `z_at` / `z_v` features, then upload `mosei_features_shard3.zip` to the shared Drive folder.

### One-time setup (leader does this once)
1. Create a shared Drive folder and share it with all 4 members.
2. Upload `segments.rar` (the 6,277 MOSEI clips) into it.
3. Everyone's `DRIVE_ROOT` (Cell 1) must point to that same folder.

### Each person
1. Runtime → Change runtime type → **T4 GPU**
2. Run all cells, top to bottom (~10-15 min for a quarter)
3. Confirm the final cell prints **shard complete** and the zip landed on Drive


## Cell 1 — Config (already set for your shard — don't change SHARD_INDEX)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  PERSON 4 — SHARD 3 of 4.  Do not edit SHARD_INDEX.
# ═══════════════════════════════════════════════════════════════════
SHARD_INDEX = 3
NUM_SHARDS  = 4

DRIVE_ROOT  = "/content/drive/MyDrive/DeepSentinel_data"          # SHARED folder — SAME for all 4 people
DRIVE_RAR_PATH   = DRIVE_ROOT + "/segments.rar"   # uploaded once by the leader
DRIVE_OUTPUT_DIR = DRIVE_ROOT                     # shard zips go here

REPO_URL    = "https://github.com/gjvlio/emotion-based-multimodal-deepfake-detector.git"
REPO_BRANCH = "feat/webapp-integration"


## Cell 2 — Mount Drive

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
assert os.path.exists(DRIVE_RAR_PATH), f'segments.rar not found at {DRIVE_RAR_PATH} — check DRIVE_ROOT'
print('Drive OK. Output →', DRIVE_OUTPUT_DIR)

## Cell 3 — Install tools + deps

In [ ]:
!apt-get update -qq && apt-get install -y -qq unrar ffmpeg
!pip install -q openai-whisper transformers timm insightface onnxruntime-gpu librosa soundfile torchaudio
print('Deps installed.')

## Cell 4 — Extract the segments RAR

In [ ]:
import subprocess
from pathlib import Path
SEGMENTS_DIR = '/content/mosei_segments'
os.makedirs(SEGMENTS_DIR, exist_ok=True)
print(f'Extracting {DRIVE_RAR_PATH} ({os.path.getsize(DRIVE_RAR_PATH)/1e9:.2f} GB)...')
r = subprocess.run(['unrar','x','-y',DRIVE_RAR_PATH,SEGMENTS_DIR+'/'], capture_output=True, text=True)
if r.returncode != 0:
    print(r.stderr[-1500:]); raise RuntimeError('unrar failed')
mp4s = list(Path(SEGMENTS_DIR).rglob('*.mp4'))
assert mp4s, 'no MP4s after extract'
ACTUAL_SEGMENTS_DIR = str(mp4s[0].parent)
print(f'{len(mp4s)} segments at {ACTUAL_SEGMENTS_DIR}')

## Cell 5 — Clone repo

In [ ]:
REPO_DIR = '/content/thesis'
if os.path.exists(REPO_DIR):
    subprocess.run(['git','-C',REPO_DIR,'pull'], check=True)
else:
    subprocess.run(['git','clone','--branch',REPO_BRANCH,'--depth','1',REPO_URL,REPO_DIR], check=True)
os.chdir(REPO_DIR)
import sys; sys.path.insert(0, REPO_DIR)
print('Repo @', REPO_BRANCH, '| CWD', os.getcwd())

## Cell 6 — Build MOSEI manifest + isolate MOSEI-only

In [ ]:
import pandas as pd
for d in ['data/preprocessed/features/z_at','data/preprocessed/features/z_v',
          'data/preprocessed/audio','data/preprocessed/transcripts',
          'data/processed/mosei_manifests']:
    Path(d).mkdir(parents=True, exist_ok=True)
segs = sorted(Path(ACTUAL_SEGMENTS_DIR).glob('*.mp4'))
pd.DataFrame([{'clip_id':v.stem,'video_path':str(v.resolve())} for v in segs])\
  .to_csv('data/processed/mosei_manifests/mosei_real.csv', index=False)
print(f'manifest: {len(segs)} clips')
# remove other manifests so preprocess_all.py targets MOSEI only
for f in ['data/processed/meld_manifests/meld_real.csv','data/raw/MUStARD/repo/data/sarcasm_data.json',
          'data/synthetic/track1_fakes/metadata.csv','data/synthetic/track2_fakes/metadata.csv',
          'data/synthetic/track3_fakes/metadata.csv','data/synthetic/track4_fakes/metadata.csv']:
    p = Path(f)
    if p.exists(): p.unlink()
print('MOSEI-only.')

## Cell 7 — Verify GPU

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(), '-', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
from src.preprocessing.pipeline import PreprocessingPipeline  # import check
print('repo imports OK')

## Cell 8 — Run preprocessing (YOUR shard only)

Processes every 4th clip starting at index 3. Resume-safe: re-run if it disconnects and it skips what you already did.

In [ ]:
!python scripts/preprocess_all.py --device cuda --num_shards 4 --shard 3 2>&1

## Cell 9 — Verify + upload your shard zip to Drive

In [ ]:
import shutil, json, time
z_at = list(Path('data/preprocessed/features/z_at').glob('*.pt'))
z_v  = list(Path('data/preprocessed/features/z_v').glob('*.pt'))
failed_log = Path('data/preprocessed/failed_clips.txt')
n_failed = len(failed_log.read_text().split()) if failed_log.exists() else 0
print(f'z_at={len(z_at)}  z_v={len(z_v)}  failed={n_failed}')
assert z_at and z_v, 'no features produced — check Cell 8 output'
# zip features → Drive (per-shard name, no collision with other people)
zip_local = '/content/mosei_features_shard3'
shutil.make_archive(zip_local, 'zip', 'data/preprocessed', 'features')
dst = os.path.join(DRIVE_OUTPUT_DIR, 'mosei_features_shard3.zip')
shutil.copy2(zip_local + '.zip', dst)
status = {'person': 4, 'shard': 3, 'z_at': len(z_at), 'z_v': len(z_v),
          'failed': n_failed, 'finished_at': time.strftime('%Y-%m-%d %H:%M:%S')}
json.dump(status, open(os.path.join(DRIVE_OUTPUT_DIR, f'shard{SHARD_INDEX}_status.json'),'w'), indent=2)
print('\n=== SHARD 3 COMPLETE ===')
print('uploaded →', dst, f'({os.path.getsize(dst)/1e6:.0f} MB)')
print('tell the leader your shard is done.')